In [2]:
import pandas as pd

akas = pd.read_csv("data/title.akas.tsv", sep="\t")
basics = pd.read_csv("data/title.basics.tsv", sep="\t")
crew = pd.read_csv("data/title.crew.tsv", sep="\t")
principals = pd.read_csv("data/title.principals.tsv", sep="\t")
names = pd.read_csv("data/name.basics.tsv", sep="\t")

In [3]:
akas.head()

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
0,tt0000001,1,Carmencita,\N,\N,original,\N,1
1,tt0000001,2,Carmencita,DE,\N,\N,literal title,0
2,tt0000001,3,Carmencita,US,\N,imdbDisplay,\N,0
3,tt0000001,4,Carmencita - spanyol tánc,HU,\N,imdbDisplay,\N,0
4,tt0000001,5,Καρμενσίτα,GR,\N,imdbDisplay,\N,0


In [4]:
akas = akas.replace("\\N", pd.NA)
filtered = akas[akas["region"].isin(["PK", "BD"])]
filtered.head()

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle
69941,tt0017224,14,"Oh, Baby!",PK,en,imdbDisplay,<NA>,0
71164,tt0017421,14,Spangles,PK,en,imdbDisplay,<NA>,0
71174,tt0017421,3,Spangles,BD,en,imdbDisplay,<NA>,0
75318,tt0018033,9,It,PK,en,imdbDisplay,<NA>,0
79822,tt0018662,10,Junak prerije,BD,bn,<NA>,literal title,0


In [5]:
filtered["region"].value_counts()

region
PK    6956
BD    5839
Name: count, dtype: int64

In [6]:
movies = basics[basics["titleType"] == "movie"]
films = filtered.merge(
    movies,
    left_on="titleId",
    right_on="tconst"
)

In [7]:
films[["primaryTitle", "startYear", "region", "language"]].head()

,primaryTitle,startYear,region,language
0,"Oh, Baby!",1926,PK,en
1,Spangles,1926,PK,en
2,Spangles,1926,BD,en
3,It,1927,PK,en
4,Avalanche,1928,BD,bn


In [8]:
films = films.merge(
    crew[["tconst", "directors", "writers"]],
    on="tconst",
    how="left"
)

films[["primaryTitle", "startYear", "region", "language", "directors", "writers"]].head()


,primaryTitle,startYear,region,language,directors,writers
0,"Oh, Baby!",1926,PK,en,nm0461295,"nm0388554,nm0461295"
1,Spangles,1926,PK,en,nm0175410,"nm0720805,nm0388919,nm0047889,nm0031033"
2,Spangles,1926,BD,en,nm0175410,"nm0720805,nm0388919,nm0047889,nm0031033"
3,It,1927,PK,en,"nm0046082,nm0903049","nm0323325,nm0521002,nm0510024,nm0547938"
4,Avalanche,1928,BD,bn,nm0112897,"nm0340719,nm0542534,nm0747844,nm0591791"


In [9]:
film_people = films.merge(
    principals[["tconst", "nconst", "category", "job", "characters"]],
    on="tconst",
    how="left"
)

film_people.head()

,titleId,ordering,title,region,language,types,attributes,isOriginalTitle,tconst,titleType,...,startYear,endYear,runtimeMinutes,genres,directors,writers,nconst,category,job,characters
0,tt0017224,14,"Oh, Baby!",PK,en,imdbDisplay,<NA>,0,tt0017224,movie,...,1926,\N,70,"Action,Comedy,Drama",nm0461295,"nm0388554,nm0461295",nm0722372,actor,\N,"[""Billy Fitzgerald""]"
1,tt0017224,14,"Oh, Baby!",PK,en,imdbDisplay,<NA>,0,tt0017224,movie,...,1926,\N,70,"Action,Comedy,Drama",nm0461295,"nm0388554,nm0461295",nm0124877,actor,\N,"[""Jim Stone""]"
2,tt0017224,14,"Oh, Baby!",PK,en,imdbDisplay,<NA>,0,tt0017224,movie,...,1926,\N,70,"Action,Comedy,Drama",nm0461295,"nm0388554,nm0461295",nm0448195,actress,\N,"[""Dorothy Brennan""]"
3,tt0017224,14,"Oh, Baby!",PK,en,imdbDisplay,<NA>,0,tt0017224,movie,...,1926,\N,70,"Action,Comedy,Drama",nm0461295,"nm0388554,nm0461295",nm0354878,actor,\N,"[""Arthur Graham""]"
4,tt0017224,14,"Oh, Baby!",PK,en,imdbDisplay,<NA>,0,tt0017224,movie,...,1926,\N,70,"Action,Comedy,Drama",nm0461295,"nm0388554,nm0461295",nm0788281,actress,\N,"[""Mary Bond""]"


In [13]:
film_people = film_people.merge(
    names[["nconst", "primaryName", "birthYear", "deathYear", "primaryProfession"]],
    on="nconst",
    how="left"
)

In [14]:
film_people[["primaryTitle", "primaryName", "category", "birthYear", "deathYear"]].head()

,primaryTitle,primaryName,category,birthYear,deathYear
0,"Oh, Baby!",'Little Billy' Rhodes,actor,1895,1967
1,"Oh, Baby!",David Butler,actor,1894,1979
2,"Oh, Baby!",Madge Kennedy,actress,1891,1987
3,"Oh, Baby!",Creighton Hale,actor,1882,1965
4,"Oh, Baby!",Ethel Shannon,actress,1898,1951


In [15]:
film_table = film_people.groupby(
    ["tconst", "primaryTitle", "startYear", "region", "language", "directors", "writers"],
    dropna=False
).agg(
    cast_crew=("primaryName", lambda x: sorted(set(x.dropna()))),
    categories=("category", lambda x: sorted(set(x.dropna())))
).reset_index()

film_table.head()

,tconst,primaryTitle,startYear,region,language,directors,writers,cast_crew,categories
0,tt0017224,"Oh, Baby!",1926,PK,en,nm0461295,"nm0388554,nm0461295","['Little Billy' Rhodes, Al Lichtman, Arthur Ho...","[actor, actress, cinematographer, director, pr..."
1,tt0017421,Spangles,1926,BD,en,nm0175410,"nm0720805,nm0388919,nm0047889,nm0031033","[André Barlatier, Charles Becker, Frank O'Conn...","[actor, actress, cinematographer, director, wr..."
2,tt0017421,Spangles,1926,PK,en,nm0175410,"nm0720805,nm0388919,nm0047889,nm0031033","[André Barlatier, Charles Becker, Frank O'Conn...","[actor, actress, cinematographer, director, wr..."
3,tt0018033,It,1927,PK,en,"nm0046082,nm0903049","nm0323325,nm0521002,nm0510024,nm0547938","[Antonio Moreno, Carl Davis, Clara Bow, Claren...","[actor, actress, cinematographer, composer, di..."
4,tt0018662,Avalanche,1928,BD,bn,nm0112897,"nm0340719,nm0542534,nm0747844,nm0591791","[Dick Winslow, Doris Hill, Guy Oliver, Herman ...","[actor, actress, cinematographer, director, ed..."


In [16]:
person_links = film_people[[
    "nconst", "primaryName", "birthYear", "deathYear",
    "primaryTitle", "startYear", "category"
]].dropna(subset=["nconst"])

person_links["startYear"] = pd.to_numeric(person_links["startYear"], errors="coerce")

person_table = (
    person_links
    .sort_values(["nconst", "startYear"])
    .groupby("nconst")
    .first()
    .reset_index()
)

person_table = person_table.rename(columns={
    "primaryTitle": "firstFilmAssociated",
    "startYear": "firstFilmYear"
})

person_table["gender"] = "Not available in IMDb public dataset"

person_table.head()

,nconst,primaryName,birthYear,deathYear,firstFilmAssociated,firstFilmYear,category,gender
0,nm0000006,Ingrid Bergman,1915,1982,Gaslight,1944.0,actress,Not available in IMDb public dataset
1,nm0000007,Humphrey Bogart,1899,1957,The Oklahoma Kid,1939.0,actor,Not available in IMDb public dataset
2,nm0000008,Marlon Brando,1924,2004,The Godfather,1972.0,actor,Not available in IMDb public dataset
3,nm0000010,James Cagney,1899,1986,Frisco Kid,1935.0,actor,Not available in IMDb public dataset
4,nm0000011,Gary Cooper,1901,1961,It,1927.0,actor,Not available in IMDb public dataset


In [17]:
film_table.to_csv("pakistan_bangladesh_films.csv", index=False)
person_table.to_csv("pakistan_bangladesh_people.csv", index=False)

In [18]:
akas["region"].unique()

array([<NA>, 'DE', 'US', 'HU', 'GR', 'RU', 'UA', 'JP', 'RO', 'FR', 'GB',
       'ES', 'CA', 'PT', 'MX', 'AU', 'EE', 'DK', 'IT', 'AR', 'FI', 'UY',
       'PL', 'BG', 'RS', 'BR', 'TR', 'SK', 'XWW', 'SUHH', 'TW', 'XEU',
       'CZ', 'SE', 'NZ', 'KZ', 'NO', 'XYU', 'AT', 'VE', 'CSHH', 'SI',
       'IN', 'NL', 'LT', 'HR', 'CN', 'CO', 'IR', 'ZA', 'IE', 'SG', 'BE',
       'CH', 'VN', 'XWG', 'CL', 'PH', 'DZ', 'BF', 'HK', 'XSA', 'GL', 'PN',
       'IS', 'PR', 'DDDE', 'EG', 'IL', 'XKO', 'EC', 'BW', 'JM', 'LV',
       'KR', 'PE', 'UZ', 'AZ', 'BY', 'GE', 'BA', 'DO', 'ID', 'TH', 'AE',
       'PA', 'TJ', 'XSI', 'MY', 'XAS', 'PK', 'BD', 'CU', 'AL', 'LU', 'BO',
       'NG', 'YUCS', 'GT', 'PY', 'SV', 'CR', 'KP', 'BUMM', 'XPI', 'BJ',
       'CM', 'MA', 'MN', 'LI', 'TN', 'FO', 'ME', 'MZ', 'MK', 'BM', 'MD',
       'LB', 'IQ', 'PS', 'NE', 'HT', 'AM', 'CI', 'LK', 'NP', 'QA', 'SY',
       'TO', 'CG', 'SN', 'GH', 'JO', 'TM', 'KG', 'GN', 'VDVN', 'TD', 'SO',
       'SD', 'MC', 'TT', 'GA', 'BS', 'LY', 'AO', 'KH',